# 01 · Situation identification with BGE-M3 (owner: Taneesha Badhe)

Section 8.1: BGE-M3 (2024) places all four languages in one meaning space. This notebook
1. embeds the training queries, 2. trains a light classifier (logistic regression on the dense vectors) and a
nearest-prototype classifier, 3. reports accuracy and macro-F1 **per language** with the gap against English,
4. compares with the TF-IDF + LR and BM25 baselines. Tune on `dev` only; report on the hand-written `test` set.

In [ ]:
!pip -q install FlagEmbedding scikit-learn rank-bm25

In [ ]:
# Clone the repo (or upload it) and move into it.
!git clone https://github.com/<owner>/firstaid-assistant.git 2>/dev/null || true
%cd firstaid-assistant/backend
!python scripts/generate_queries.py

In [ ]:
import json
def load(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train, dev = load("data/queries/train.jsonl"), load("data/queries/dev.jsonl")
try:
    test = load("data/queries/test.jsonl")
except FileNotFoundError:
    test = []
print(len(train), len(dev), len(test))

In [ ]:
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
enc = lambda rows: model.encode([r["text"] for r in rows], batch_size=64)["dense_vecs"]
Xtr, Xdev = enc(train), enc(dev)
ytr, ydev = [r["situation"] for r in train], [r["situation"] for r in dev]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
clf = LogisticRegression(max_iter=3000, C=4.0).fit(Xtr, ytr)
def report(rows, X, name):
    pred = clf.predict(X)
    en = None
    for lang in ["en", "hi", "hinglish", "mr"]:
        idx = [i for i, r in enumerate(rows) if r["lang"] == lang]
        if not idx: continue
        g = [rows[i]["situation"] for i in idx]; p = [pred[i] for i in idx]
        acc, mf1 = accuracy_score(g, p), f1_score(g, p, average="macro")
        en = en if en is not None else acc
        print(f"{name} {lang:9s} acc={acc:.3f} macroF1={mf1:.3f} gap={acc-en:+.3f}")
report(dev, Xdev, "dev")

In [ ]:
# Save for MODEL_MODE=real: the backend's RealSituationClassifier uses recognition-text prototypes by default;
# to use this trained head instead, pickle it next to the backend and load it in app/models/real.
import pickle, os
os.makedirs("../training/out", exist_ok=True)
pickle.dump(clf, open("../training/out/bge_m3_situation_lr.pkl", "wb"))